In [1]:
from pathlib import Path
import json
import random
import re
import unicodedata

In [2]:
EXCERPT_ID = "excerpt_001"

PROJECT_ROOT = Path("/workspace/storyfx")

DISCRETE_SPEC_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "composition_plans"
    / f"pred_{EXCERPT_ID}_discrete_sfx_specification.json"
)

FA_PARSED_PATH = (
    PROJECT_ROOT
    / "data"
    / EXCERPT_ID
    / "alignment"
    / f"{EXCERPT_ID}_fa_parsed.txt"
)

SEGMENT_SCHEDULE_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "schedules"
    / f"{EXCERPT_ID}_segment_schedule.json"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "schedules"
    / f"{EXCERPT_ID}_discrete_sfx_schedule.json"
)


print("Discrete specification:", DISCRETE_SPEC_PATH)
print("Forced alignment:", FA_PARSED_PATH)
print("Segment schedule:", SEGMENT_SCHEDULE_PATH)
print("Output:", OUTPUT_PATH)

Discrete specification: /workspace/storyfx/outputs/composition_plans/pred_excerpt_001_discrete_sfx_specification.json
Forced alignment: /workspace/storyfx/data/excerpt_001/alignment/excerpt_001_fa_parsed.txt
Segment schedule: /workspace/storyfx/outputs/schedules/excerpt_001_segment_schedule.json
Output: /workspace/storyfx/outputs/schedules/excerpt_001_discrete_sfx_schedule.json


In [3]:
def normalize_text(text):
    text = unicodedata.normalize(
        "NFKC",
        text,
    )

    text = (
        text
        .replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
    )

    return text.lower()


def lexical_tokens(text):
    text = normalize_text(text)

    return re.findall(
        r"[a-z0-9]+(?:'[a-z0-9]+)?",
        text,
    )

In [4]:
def load_fa_words(path):
    text = Path(path).read_text(
        encoding="utf-8"
    ).strip()

    words = []

    for item in text.split():
        if "|" not in item:
            continue

        word, end_time = item.rsplit(
            "|",
            1,
        )

        words.append(
            {
                "word": word,
                "end_time": float(end_time),
            }
        )

    for i, word in enumerate(words):
        if i == 0:
            word["start_time"] = 0.0
        else:
            word["start_time"] = words[
                i - 1
            ]["end_time"]

    return words


fa_words = load_fa_words(
    FA_PARSED_PATH
)

print(
    "Aligned words:",
    len(fa_words),
)

Aligned words: 971


In [10]:
def find_anchor(
    anchor,
    fa_words,
    start_index=0,
):
    target = lexical_tokens(anchor)

    if not target:
        raise ValueError(
            f"Anchor contains no matchable words: {anchor}"
        )

    for start in range(
        start_index,
        len(fa_words),
    ):
        collected = []
        end = start

        while (
            end < len(fa_words)
            and len(collected) < len(target)
        ):
            collected.extend(
                lexical_tokens(
                    fa_words[end]["word"]
                )
            )

            end += 1

        if collected == target:
            return start, end - 1

    raise ValueError(
        f"Could not match anchor:\n"
        f"{anchor}\n\n"
        f"Normalized tokens:\n"
        f"{target}"
    )

In [5]:
RANDOM_SEED = 42

rng = random.Random(
    RANDOM_SEED
)


DURATION_RANGES = {
    "impulse": (0.8, 1.0),
    "brief": (1.1, 2.5),
    "prolonged": (2.6, 3.5),
}


def sample_duration_time(
    duration_class,
    rng,
):
    if duration_class not in DURATION_RANGES:
        raise ValueError(
            f"Unknown duration class: "
            f"{duration_class}"
        )

    minimum, maximum = (
        DURATION_RANGES[
            duration_class
        ]
    )

    duration_time = rng.uniform(
        minimum,
        maximum,
    )

    return round(
        duration_time,
        2,
    )

In [6]:
def interval_overlap(
    start_a,
    end_a,
    start_b,
    end_b,
):
    return max(
        0.0,
        min(end_a, end_b)
        - max(start_a, start_b),
    )

In [7]:
def find_parent_segment(
    anchor_time,
    duration_time,
    segment_schedule,
):
    sfx_start = anchor_time

    sfx_end = (
        anchor_time
        + duration_time
    )

    best_segment = None
    best_overlap = 0.0

    for segment in segment_schedule:
        overlap = interval_overlap(
            sfx_start,
            sfx_end,
            segment["start_time"],
            segment["end_time"],
        )

        if overlap > best_overlap:
            best_overlap = overlap
            best_segment = segment

    if best_segment is None:
        raise ValueError(
            "Could not assign parent segment "
            f"for SFX interval "
            f"{sfx_start:.2f}–{sfx_end:.2f}"
        )

    return best_segment

In [8]:
def quantify_discrete_sfx(
    discrete_spec_path,
    fa_parsed_path,
    segment_schedule_path,
    random_seed=42,
):
    discrete_sfx = json.loads(
        Path(discrete_spec_path).read_text(
            encoding="utf-8"
        )
    )

    segment_schedule = json.loads(
        Path(segment_schedule_path).read_text(
            encoding="utf-8"
        )
    )

    fa_words = load_fa_words(
        fa_parsed_path
    )

    rng = random.Random(
        random_seed
    )

    quantified_sfx = []

    search_index = 0

    for sfx in discrete_sfx:
        anchor = sfx["anchor"]

        print(
            f"\n--- "
            f"{sfx['discrete_sfx_id']} "
            f"---"
        )

        print(
            "Anchor:",
            anchor,
        )

        # ----------------------------------------
        # Find anchor
        # ----------------------------------------

        anchor_first, anchor_last = (
            find_anchor(
                anchor,
                fa_words,
                start_index=search_index,
            )
        )

        print(
            "Matched:",
            anchor_first,
            "->",
            anchor_last,
        )

        print(
            "Matched words:",
            " ".join(
                word["word"]
                for word in fa_words[
                    anchor_first:
                    anchor_last + 1
                ]
            ),
        )

        anchor_time = fa_words[
            anchor_last
        ]["end_time"]

        # ----------------------------------------
        # Sample playback duration
        # ----------------------------------------

        duration_time = (
            sample_duration_time(
                sfx["duration"],
                rng,
            )
        )

        # ----------------------------------------
        # Parent segment
        # ----------------------------------------

        parent = find_parent_segment(
            anchor_time,
            duration_time,
            segment_schedule,
        )

        quantified = {
            **sfx,
            "anchor_time": round(
                anchor_time,
                2,
            ),
            "duration_time": duration_time,
            "parent_segment": parent[
                "segment_id"
            ],
            "reverb_profile": parent[
                "reverb_profile"
            ],
        }

        quantified_sfx.append(
            quantified
        )

        # Anchors should occur in narrative order.
        search_index = (
            anchor_last + 1
        )

        print(
            "Anchor time:",
            quantified["anchor_time"],
        )

        print(
            "Duration time:",
            quantified["duration_time"],
        )

        print(
            "Parent segment:",
            quantified["parent_segment"],
        )

        print(
            "Reverb:",
            quantified[
                "reverb_profile"
            ],
        )

    return quantified_sfx

In [11]:
discrete_sfx_schedule = (
    quantify_discrete_sfx(
        DISCRETE_SPEC_PATH,
        FA_PARSED_PATH,
        SEGMENT_SCHEDULE_PATH,
        random_seed=42,
    )
)

discrete_sfx_schedule


--- discrete_sfx000 ---
Anchor: Corbin barely heard the chimes
Matched: 227 -> 231
Matched words: corbin barely heard the chimes
Anchor time: 95.24
Duration time: 2.0
Parent segment: segment_001
Reverb: indoor_medium_absorbed_regular

--- discrete_sfx001 ---
Anchor: he plunged himself into a musty mink cloak
Matched: 247 -> 254
Matched words: he plunged himself into a musty mink cloak
Anchor time: 104.19
Duration time: 1.14
Parent segment: segment_001
Reverb: indoor_medium_absorbed_regular

--- discrete_sfx002 ---
Anchor: snatched the burning lantern off the dresser
Matched: 256 -> 262
Matched words: snatched the burning lantern off the dresser
Anchor time: 107.11
Duration time: 1.49
Parent segment: segment_002
Reverb: outdoor_open

--- discrete_sfx003 ---
Anchor: his anxious strides carried him down a rain-lashed colonnade
Matched: 266 -> 275
Matched words: his anxious strides carried him down a rain lashed colonnade
Anchor time: 112.36
Duration time: 2.8
Parent segment: segment_002


[{'discrete_sfx_id': 'discrete_sfx000',
  'anchor': 'Corbin barely heard the chimes',
  'caption': 'grandfather clock chiming',
  'duration': 'brief',
  'transmission_profile': None,
  'anchor_time': 95.24,
  'duration_time': 2.0,
  'parent_segment': 'segment_001',
  'reverb_profile': 'indoor_medium_absorbed_regular'},
 {'discrete_sfx_id': 'discrete_sfx001',
  'anchor': 'he plunged himself into a musty mink cloak',
  'caption': 'heavy fur cloak rustling',
  'duration': 'brief',
  'transmission_profile': None,
  'anchor_time': 104.19,
  'duration_time': 1.14,
  'parent_segment': 'segment_001',
  'reverb_profile': 'indoor_medium_absorbed_regular'},
 {'discrete_sfx_id': 'discrete_sfx002',
  'anchor': 'snatched the burning lantern off the dresser',
  'caption': 'metal lantern abruptly lifted',
  'duration': 'brief',
  'transmission_profile': None,
  'anchor_time': 107.11,
  'duration_time': 1.49,
  'parent_segment': 'segment_002',
  'reverb_profile': 'outdoor_open'},
 {'discrete_sfx_id': '

In [12]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_PATH.write_text(
    json.dumps(
        discrete_sfx_schedule,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    f"Discrete SFX schedule saved to: "
    f"{OUTPUT_PATH}"
)

Discrete SFX schedule saved to: /workspace/storyfx/outputs/schedules/excerpt_001_discrete_sfx_schedule.json
